In [14]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import SAGEConv, Node2Vec
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dataset = Planetoid(root="data/Cora", name="Cora")
data = dataset[0]

C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\torch_geometric\data\dataset.py:238: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  if osp.exists(f) and torch.lo

In [15]:
transform = RandomLinkSplit(
    num_val=0.05,
    num_test=0.1,
    is_undirected=True,
    add_negative_train_samples=False
)

train_data, val_data, test_data = transform(data)

train_data = train_data.to(device)
val_data = val_data.to(device)
test_data = test_data.to(device)

In [32]:
# DeepWalk is the same as Node2Vec with p=1 and q=1

dw_model = Node2Vec(
    train_data.edge_index,
    embedding_dim=64,
    walk_length=21,
    context_size=9,
    walks_per_node=12,
    num_negative_samples=1,
    p=1,
    q=1,
    sparse=True
).to(device)

loader = dw_model.loader(batch_size=256, shuffle=True)
dw_optimizer = torch.optim.SparseAdam(dw_model.parameters(), lr=0.01)


In [33]:
# =====================================
# 6. Train DeepWalk
# =====================================

def train_deepwalk():

    dw_model.train()
    total_loss = 0

    for pos_rw, neg_rw in loader:

        dw_optimizer.zero_grad()

        loss = dw_model.loss(pos_rw.to(device), neg_rw.to(device))

        loss.backward()
        dw_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)



@torch.no_grad()
def evaluate_deepwalk(data):

    dw_model.eval()

    z = dw_model()
    z = F.normalize(z, p=2, dim=1)

    out = (z[data.edge_label_index[0]] * z[data.edge_label_index[1]]).sum(dim=-1).sigmoid()

    y_true = data.edge_label.cpu().numpy()
    y_pred = out.cpu().numpy()

    auc = roc_auc_score(y_true, y_pred)
    ap = average_precision_score(y_true, y_pred)

    return auc, ap



print("\nTraining DeepWalk")


for epoch in range(1, 81):

    loss = train_deepwalk()

    if epoch % 5 == 0:

        val_auc, val_ap = evaluate_deepwalk(val_data)

        print(f"Epoch {epoch:03d} | Loss {loss:.4f} | Val AUC {val_auc:.4f} | Val AP {val_ap:.4f}")
        dw_auc, dw_ap = evaluate_deepwalk(test_data)
        print("AUC:", dw_auc)
        print("AP :", dw_ap)


dw_auc, dw_ap = evaluate_deepwalk(test_data)

print("\nDeepWalk Test")
print("AUC:", dw_auc)
print("AP :", dw_ap)


Training DeepWalk
Epoch 005 | Loss 2.8964 | Val AUC 0.6432 | Val AP 0.5873
AUC: 0.6191197174223793
AP : 0.5736553867158329
Epoch 010 | Loss 1.7322 | Val AUC 0.7236 | Val AP 0.6755
AUC: 0.6946285767780822
AP : 0.6590934154355913
Epoch 015 | Loss 1.2467 | Val AUC 0.7839 | Val AP 0.7714
AUC: 0.758628015079448
AP : 0.7561689348327075
Epoch 020 | Loss 1.0385 | Val AUC 0.8307 | Val AP 0.8393
AUC: 0.806487619225936
AP : 0.8267965728845417
Epoch 025 | Loss 0.9444 | Val AUC 0.8551 | Val AP 0.8744
AUC: 0.8319224855884693
AP : 0.8590907538695527
Epoch 030 | Loss 0.8951 | Val AUC 0.8636 | Val AP 0.8850
AUC: 0.844783944060577
AP : 0.8731650999899688
Epoch 035 | Loss 0.8676 | Val AUC 0.8676 | Val AP 0.8926
AUC: 0.8516467491691542
AP : 0.880185337771156
Epoch 040 | Loss 0.8498 | Val AUC 0.8721 | Val AP 0.8971
AUC: 0.8589164257243572
AP : 0.8859371104731768
Epoch 045 | Loss 0.8377 | Val AUC 0.8726 | Val AP 0.8993
AUC: 0.8593485015968805
AP : 0.8879581927568914
Epoch 050 | Loss 0.8296 | Val AUC 0.8722

In the above cell after running a few times the training with different configurations and monitoring the outputs I picked the best number of epochs and configuration.

In [34]:
def train_deepwalk():

    dw_model.train()
    total_loss = 0

    for pos_rw, neg_rw in loader:

        dw_optimizer.zero_grad()

        loss = dw_model.loss(pos_rw.to(device), neg_rw.to(device))

        loss.backward()
        dw_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)



@torch.no_grad()
def evaluate_deepwalk(data):

    dw_model.eval()

    z = dw_model()
    z = F.normalize(z, p=2, dim=1)

    out = (z[data.edge_label_index[0]] * z[data.edge_label_index[1]]).sum(dim=-1).sigmoid()

    y_true = data.edge_label.cpu().numpy()
    y_pred = out.cpu().numpy()

    auc = roc_auc_score(y_true, y_pred)
    ap = average_precision_score(y_true, y_pred)

    return auc, ap



print("\nTraining DeepWalk")

num_runs = 50
test_aucs = []
test_aps = []

for run in range(1, num_runs + 1):

    dw_model = Node2Vec(
    train_data.edge_index,
    embedding_dim=64,
    walk_length=21,
    context_size=9,
    walks_per_node=12,
    num_negative_samples=1,
    p=1,
    q=1,
    sparse=True
).to(device)

    loader = dw_model.loader(batch_size=256, shuffle=True)
    dw_optimizer = torch.optim.SparseAdam(dw_model.parameters(), lr=0.01)
    for epoch in range(1, 81):
        loss = train_deepwalk()
        
    dw_auc, dw_ap = evaluate_deepwalk(test_data)
    test_aucs.append(dw_auc)
    test_aps.append(dw_ap)
    print(f"Run {run:02d} Completed - Test AUC: {dw_auc:.4f}, Test AP: {dw_ap:.4f}")

print(f"\n--- Final Results over {num_runs} runs ---")
print(f"Test AUC: {np.mean(test_aucs):.4f} ± {np.std(test_aucs):.4f}")
print(f"Test AP:  {np.mean(test_aps):.4f} ± {np.std(test_aps):.4f}")


Training DeepWalk
Run 01 Completed - Test AUC: 0.8591, Test AP: 0.8865
Run 02 Completed - Test AUC: 0.8575, Test AP: 0.8849
Run 03 Completed - Test AUC: 0.8698, Test AP: 0.8897
Run 04 Completed - Test AUC: 0.8599, Test AP: 0.8864
Run 05 Completed - Test AUC: 0.8642, Test AP: 0.8904
Run 06 Completed - Test AUC: 0.8559, Test AP: 0.8860
Run 07 Completed - Test AUC: 0.8610, Test AP: 0.8868
Run 08 Completed - Test AUC: 0.8646, Test AP: 0.8882
Run 09 Completed - Test AUC: 0.8752, Test AP: 0.8963
Run 10 Completed - Test AUC: 0.8663, Test AP: 0.8913
Run 11 Completed - Test AUC: 0.8674, Test AP: 0.8942
Run 12 Completed - Test AUC: 0.8611, Test AP: 0.8905
Run 13 Completed - Test AUC: 0.8518, Test AP: 0.8812
Run 14 Completed - Test AUC: 0.8684, Test AP: 0.8903
Run 15 Completed - Test AUC: 0.8587, Test AP: 0.8943
Run 16 Completed - Test AUC: 0.8579, Test AP: 0.8872
Run 17 Completed - Test AUC: 0.8611, Test AP: 0.8904
Run 18 Completed - Test AUC: 0.8675, Test AP: 0.8892
Run 19 Completed - Test AUC